In [2]:
!pip install -q -U langgraph langchain_google_genai trustcall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 kB 12.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.4 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.17 which is incompatible.


In [3]:
import pandas as pd
from langchain_core.tools import tool
import google.generativeai as genai
import time
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from uuid import uuid4
import os
import getpass
from trustcall import create_extractor

In [ ]:
GEMINI_API_KEY = getpass.getpass()
genai.configure(api_key=GEMINI_API_KEY)

··········


In [4]:
PROMPT_VIDEO = """
Você é um especialista em segurança do trabalho.
Seu objetivo é analisar microfilmagens, com ou sem áudio, ou,
na ausência destas, utilizar as informações fornecidas sobre o segmento de atuação
da empresa para identificar e classificar riscos ocupacionais nas categorias:
físicos, químicos, biológicos, de acidentes, ergonômicos e psicossociais.

Para cada risco identificado, você DEVE fornecer as seguintes informações em formato estruturado:
- Risco: nome do risco identificado
- Fonte_Geradora: a origem ou fonte do risco
- Agente: tipo do risco (Físico, Químico, Biológico, Acidente, Ergonômico, Psicossocial)
- Medidas_de_Controle: ações recomendadas para mitigar o risco
- Severidade: IRREVERSÍVEL SEVERO, LEVE, REVERSÍVEL SEVERO, INCAPACITANTE OU FATAL, ALTAMENTE CATASTRÓFICO
- Probabilidade: IMPROVÁVEL, POSSÍVEL, MUITO IMPROVÁVEL, MUITO PROVÁVEL, PROVÁVEL
- Nivel_de_Risco: RISCO MÉDIO, RISCO BAIXO, RISCO IRRELEVANTE, RISCO ALTO, RISCO CRÍTICO

Instruções:

Identificação de Riscos:

Riscos Físicos: Observe a presença de agentes como ruído, vibração, temperaturas extremas (calor ou frio),
umidade, pressões anormais, radiações ionizantes e não ionizantes

Riscos Químicos: Identifique a exposição a substâncias químicas nocivas, incluindo poeiras, fumos, névoas,
neblinas, gases e vapores que possam ser inalados ou absorvidos pela pele

Riscos Biológicos: Detecte a presença de agentes biológicos como bactérias, vírus, fungos,
parasitas e outros microrganismos que possam causar doenças.

Riscos de Acidentes: Verifique situações que possam resultar em quedas, cortes, amputações,
choques elétricos, incêndios, explosões e outros acidentes

Riscos Ergonômicos: Avalie posturas inadequadas, esforços repetitivos, levantamento de peso,
jornadas prolongadas e outros fatores que possam afetar a saúde musculoesquelética

Riscos Psicossociais: Identifique sinais de estresse, assédio moral, sobrecarga de trabalho,
falta de controle sobre as tarefas, entre outros fatores que possam afetar a saúde mental dos trabalhadores

Importante: Ao final da sua análise, apresente os riscos encontrados em formato de tabela com as colunas listadas acima. A tabela deve ser formatada de maneira que cada linha represente um risco diferente. Apresente a tabela com o título "INVENTÁRIO DE RISCOS:".
"""

In [5]:
# Prompt de sistema para o assistente que processará o inventário
PROMPT_SYS_MESSAGE = """
Você é um especialista em segurança do trabalho.
Sua tarefa é extrair informações do inventário de riscos e inserir cada risco no dataframe.

Para cada risco identificado no inventário, você DEVE chamar a ferramenta add_data_to_df com os seguintes parâmetros:
- risco: o nome do risco identificado
- fonte_geradora: a origem ou fonte do risco
- agente: o tipo do risco (Físico, Químico, Biológico, Acidente, Ergonômico, Psicossocial)
- medidas_de_controle: as ações recomendadas para mitigar o risco
- severidade: a classificação de severidade do risco
- probabilidade: a classificação de probabilidade do risco
- nivel_de_risco: a classificação geral do risco

Procure por seções ou tabelas no texto que contenham "INVENTÁRIO DE RISCOS:" ou listagens de riscos.
Você deve processar CADA RISCO INDIVIDUALMENTE, um por vez, fazendo uma chamada separada da ferramenta para cada risco.
Não pule nenhum risco identificado.

Se não conseguir extrair alguma informação, preencha com "Não especificado" em vez de deixar em branco.
"""

In [6]:
def analyze_video(video_file_name,
                  display_name:str="video.mp4",
                  model_name:str="gemini-2.0-flash"):

    fileList = genai.list_files(page_size=100)
    video_file = next((f for f in fileList if f.display_name == display_name), None)
    if video_file is None:
        print(f"Uploading file...")
        # for video_file_name in videos_file_name:
        video_file = genai.upload_file(path=video_file_name,
                                       display_name=display_name,
                                       resumable=True)

        print(f"Completed upload: {video_file.uri}")
    else:
        print(f"File URI: {video_file.uri}")

    # Check the state of the uploaded file.
    print("Processing file...")
    while video_file.state.name == "PROCESSING":
        time.sleep(5)
        video_file = genai.get_file(video_file.name)

    if video_file.state.name == "FAILED":
        raise ValueError(video_file.state.name)

    # Generate content using the uploaded file.
    model = genai.GenerativeModel(model_name=model_name)
    print("Making LLM inference request...")
    response = model.generate_content([video_file, PROMPT_VIDEO],
                                      request_options={"timeout": 600})
    return response.text

In [7]:
# Função para adicionar dados ao DataFrame
@tool
def add_data_to_df(
    risco: str,
    fonte_geradora: str,
    agente: str,
    medidas_de_controle: str,
    severidade: str,
    probabilidade: str,
    nivel_de_risco: str
    ) -> str:
    """Adiciona dados ao dataframe
      Parameters:
      risco (str): Potencial de dano de cada risco
      fonte_geradora (str): Origem ou fonte que gera o risco no ambiente, qual material ou produto que está gerando o risco.
      agente (str): Qual tipo do risco (ex: Físico, Químico, Biológico, Acidente, Ergonômico, Psicossociais, etc.).
      medidas_de_controle (str): Ações preventivas ou corretivas para mitigar o risco.
      severidade (str): Nível de gravidade do risco (ex: IRREVERSÍVEL SEVERO, LEVE, REVERSÍVEL SEVERO, INCAPACITANTE OU FATAL, ALTAMENTE CATASTRÓFICO).
      probabilidade (str): Chance de ocorrência do risco (ex: IMPROVÁVEL, POSSÍVEL, MUITO IMPROVÁVEL, MUITO PROVÁVEL, PROVÁVEL).
      nivel_de_risco (str): Classificação geral resultante da combinação de severidade e probabilidade (ex: RISCO MÉDIO, RISCO BAIXO, RISCO IRRELEVANTE, RISCO ALTO, RISCO CRÍTICO)
    """
    path_data = os.path.join(os.path.dirname(__file__), "data.csv")

    # Ler o CSV existente
    df = pd.read_csv(path_data)

    # Criar um dicionário com os dados
    new_data = {
        "Risco": risco,
        "Fonte_Geradora": fonte_geradora,
        "Agente": agente,
        "Medidas_de_Controle": medidas_de_controle,
        "Severidade": severidade,
        "Probabilidade": probabilidade,
        "Nivel_de_Risco": nivel_de_risco
    }

    # Adicionar os dados ao dataframe
    df = pd.concat([df, pd.DataFrame([new_data])], ignore_index=True)

    # Salvar o dataframe
    df.to_csv(path_data, index=False)

    return f"Dados adicionados com sucesso! {new_data}"

In [8]:
def analyze_node(state: MessagesState):
    video = state["messages"]
    return {"messages": analyze_video(video)}

def assistant(state: MessagesState):
  llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GEMINI_API_KEY)
  trustcall_extractor = create_extractor(
      llm=llm,
      tools=[add_data_to_df])
  return {"messages": [trustcall_extractor.invoke([PROMPT_SYS_MESSAGE] + state["messages"])]}

In [ ]:
df = pd.read_csv("/content/data.csv")
df.head()

,Risco,Fonte_Geradora,Agente,Medidas_de_Controle,Severidade,Probabilidade,Nivel_de_Risco


In [45]:
def should_continue(state: MessagesState):
    # Verifica a última mensagem na lista de mensagens
    from langchain_core.messages import ToolMessage
    last_message = state["messages"][-1]

    # Verifica se a última mensagem é uma ToolMessage
    if isinstance(last_message, ToolMessage):
        error_message = "Registro já existe no inventário de riscos. Dados não adicionados."

        # Verifica o conteúdo da última mensagem da ferramenta
        if last_message.content == error_message:
            return END

    return "tools"

In [76]:
# from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from trustcall import create_extractor
import uuid


import os


# video_file_name = os.path.join(os.path.dirname(__file__), "video.mp4")

def analyze_node(state: MessagesState):
    """Analisa o vídeo enviado"""
    video_path = state["messages"][-1].content

    # Chama a função de análise de vídeo
    video_analysis = analyze_video(video_path)

    # Retorna o resultado como uma mensagem AI
    return {"messages": [video_analysis]}


def assistant(state: MessagesState):
  llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GEMINI_API_KEY)
  trustcall_extractor = create_extractor(
      llm=llm,
      tools=[add_data_to_df])
  return {"messages": [trustcall_extractor.invoke([PROMPT_SYS_MESSAGE] + state["messages"])]}

def should_continue_after_assistant(state: MessagesState):
    """Decide se chama ferramentas ou termina após o assistente."""
    last_message = state["messages"][-1]
    # Se a última mensagem for do assistente e tiver chamadas de ferramenta, vá para as ferramentas
    if isinstance(last_message, AIMessage) and hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    # Caso contrário, termine
    else:
        return END # Use END directly as the key for the next step

def should_continue_after_tools(state: MessagesState):
    """Decide se termina ou volta para o assistente após a execução das ferramentas."""
    last_message = state["messages"][-1]
    error_message = "Registro já existe no inventário de riscos. Dados não adicionados."
    if last_message.content == error_message:
        return END
    else:
        return "assistant"

def graph_builder():
    builder = StateGraph(MessagesState)

    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools=[add_data_to_df]))
    builder.add_node("analyze_node", analyze_node)

    builder.add_edge(START, "analyze_node")
    builder.add_edge("analyze_node", "assistant")

    builder.add_edge("assistant", "tools")

    builder.add_conditional_edges("tools", should_continue_after_tools,
        {"assistant": "assistant",
            END: END,
        }
    )

    return builder.compile(checkpointer=MemorySaver())

In [ ]:
from IPython.display import Image, display

graph = graph_builder()
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
# Definir ID para a thread
id = str(uuid4())
config = {"configurable": {"thread_id": id, "user_id": id}}

# Invocar o grafo com o caminho do vídeo
video_path = "/content/video.mp4"

output = graph.invoke(
    {"messages": [video_path]},
    config,
    stream_mode="values",
    debug=True
)

# Imprimir as mensagens de saída
for message in output["messages"]:
    message.pretty_print()

In [ ]:
# Carregar e mostrar o dataframe final
df = pd.read_csv("/content/data.csv")
df

,Risco,Fonte_Geradora,Agente,Medidas_de_Controle,Severidade,Probabilidade,Nivel_de_Risco


In [ ]:
df["Risco"]